# 00.5 — Transpose, and why `W^T` is everywhere

**Question:** why does attention compute `Q @ K^T` rather than `Q @ K`, and why does a transposed weight turn up in every backward-pass formula?

**Prereq:** 00.4 (inner dims match and vanish, outer survive).

**Interview one-liners**
- Transpose **does not move data**. It returns a *view* with the strides swapped. That is why it is free — and why it can make a later `.view()` fail.
- `(A @ B)^T = B^T A^T` — the order reverses. This identity is why a transposed weight appears in every backward pass.
- `Q @ K^T` exists to make the **inner dimension** `d_k`, so it vanishes and you are left with a `(T, T)` matrix of token-to-token scores.
- **`(T, T)` is the single most important shape in the curriculum.** Its size is `O(T²)` and it is *independent of `d_k`* — which is why long context is a memory problem, not a width problem.
- Entry `[i, j]` of that matrix is one dot product: query `i` against key `j`.

In [1]:
import sys
sys.path.insert(0, '..')

import torch
from common import show

torch.manual_seed(0)
print('torch', torch.__version__)

torch 2.10.0


## Experiment 1 — what transpose actually does

Rows become columns. The obvious part is the shape. The part that matters is the **strides**.

A stride is "how many elements do I skip in memory to take one step along this axis".

In [2]:
A = torch.arange(6).reshape(2, 3)

print('A            ', tuple(A.shape), ' stride', A.stride())
print(A)
print('\nA.T          ', tuple(A.T.shape), ' stride', A.T.stride(), '  <- swapped')
print(A.T)
print('\nsame storage?     ', A.data_ptr() == A.T.data_ptr())
print('A   contiguous?   ', A.is_contiguous())
print('A.T contiguous?   ', A.T.is_contiguous(), '  <- the consequence')

A             (2, 3)  stride (3, 1)
tensor([[0, 1, 2],
        [3, 4, 5]])

A.T           (3, 2)  stride (1, 3)   <- swapped
tensor([[0, 3],
        [1, 4],
        [2, 5]])

same storage?      True
A   contiguous?    True
A.T contiguous?    False   <- the consequence


### Reading the output
Stride `(3, 1)` becomes `(1, 3)` — **just swapped**. Same pointer, no copy, no data moved.
Transposing a 10 GB tensor costs nothing.

But `A.T` is **not contiguous**: walking it in row-major order now jumps around in memory.
That is the price, and it is exactly why `.view()` refuses to work on a transposed tensor
(01.5 makes that its whole subject).

> **Common misconception:** that `.T` copies. It does not. If you *need* a copy, you ask for
> one with `.contiguous()`, and then you pay for it.

## Experiment 2 — `(A @ B)^T = B^T A^T`

The order **reverses**. Verify it numerically rather than taking my word for it.

In [3]:
A = torch.randn(2, 3)
B = torch.randn(3, 4)

lhs = (A @ B).T          # (2,4) -> (4,2)
rhs = B.T @ A.T          # (4,3) @ (3,2) -> (4,2)

print('(A @ B).T  ', tuple(lhs.shape))
print('B.T @ A.T  ', tuple(rhs.shape))
print('equal?     ', torch.allclose(lhs, rhs))
print()

# and the order really does matter — A.T @ B.T is not even legal here
try:
    A.T @ B.T            # (3,2) @ (4,3)
except RuntimeError as e:
    print('A.T @ B.T  ->', str(e).split('\n')[0])

(A @ B).T   (4, 2)
B.T @ A.T   (4, 2)
equal?      True

A.T @ B.T  -> mat1 and mat2 shapes cannot be multiplied (3x2 and 4x3)


### Reading the output
`True`, and the reversed order is not optional — `A.T @ B.T` does not even have matching inner
dimensions.

**Why this identity matters later:** a linear layer computes `y = x @ W`. When gradients flow
backwards through it, the formula that falls out contains `W^T`. You have not done backprop yet
(that is Phase 1), but when you get there and wonder *"where did that transpose come from"* —
it came from here.

## Experiment 3 — why attention needs the transpose

`Q` and `K` are both `(T, d_k)`: one row per token, `d_k` numbers describing each.

We want a score for **every (query, key) pair** — how much does token `i` care about token `j`.

> **Predict first, before running the next cell.**
>
> `Q` is `(4, 8)` and `K` is `(4, 8)`.
>
> 1. `Q @ K` — does it work? If not, which two numbers clash?
> 2. `Q @ K.T` — what shape comes out, and which dimension vanished?
> 3. What is row `i`, column `j` of that result, in words?
>
> _(write your three answers here)_

In [4]:
T, d_k = 4, 8
Q = torch.randn(T, d_k)
K = torch.randn(T, d_k)

try:
    Q @ K
except RuntimeError as e:
    print('Q @ K    ->', str(e).split('\n')[0])

scores = Q @ K.T
print('\nQ   ', tuple(Q.shape))
print('K.T ', tuple(K.T.shape))
print('Q @ K.T ->', tuple(scores.shape), '   the d_k=8 vanished, T=4 survived twice')

Q @ K    -> mat1 and mat2 shapes cannot be multiplied (4x8 and 4x8)

Q    (4, 8)
K.T  (8, 4)
Q @ K.T -> (4, 4)    the d_k=8 vanished, T=4 survived twice


### Reading the output
`(4, 8) @ (8, 4) -> (4, 4)`.

The transpose exists for one reason: **to put `d_k` on the inside** so it matches, gets summed
over, and disappears. What is left is indexed by tokens on both axes.

This is 00.4's rule doing all the work — inner matches and vanishes, outer survive.

## Experiment 4 — what one entry of that matrix *is*

Claim: `scores[i, j]` is exactly one dot product — row `i` of `Q` against row `j` of `K`.
That is the 00.3 operation, unchanged.

In [5]:
i, j = 2, 0      # "token 2 querying token 0"

print('scores[2, 0]        =', scores[i, j].item())
print('dot(Q[2], K[0])     =', torch.dot(Q[i], K[j]).item())
print('identical?           ', torch.allclose(scores[i, j], torch.dot(Q[i], K[j])))
print()
print('scores =')
print(scores)
print('\nrow 2 = token 2 scored against every token:', [round(v, 3) for v in scores[2].tolist()])

scores[2, 0]        = -3.982325315475464
dot(Q[2], K[0])     = -3.982325315475464
identical?            True

scores =
tensor([[-0.8098, -0.4794,  2.3069,  1.3712],
        [ 5.2321,  3.1860, -4.0295, -6.9740],
        [-3.9823,  0.7302, -0.2628, -1.2328],
        [-4.8545, -0.9189,  0.5262,  3.5359]])

row 2 = token 2 scored against every token: [-3.982, 0.73, -0.263, -1.233]


### Reading the output
`scores[2, 0]` and `dot(Q[2], K[0])` are the same number.

So the `(T, T)` matrix is **`T²` dot products** — token `i`'s query against token `j`'s key,
for every pair. Row `i` is "everything token `i` is looking at".

Note that nothing is masked yet and nothing has been through softmax. These are raw scores.
That is where your paused `02_attention/` lesson picks up.

## Experiment 5 — the shape that decides long context

The claim worth memorising: the score matrix is `O(T²)` and **completely independent of `d_k`**.

Widen the model and the score matrix does not grow. Lengthen the sequence and it grows squared.

In [6]:
print('vary d_k, hold T=4:')
for d in (8, 64, 512):
    q, k = torch.randn(4, d), torch.randn(4, d)
    print(f'   d_k={d:4d}  ->  scores {tuple((q @ k.T).shape)}   ({(q @ k.T).numel()} entries)')

print('\nvary T, hold d_k=8:')
for t in (4, 8, 16, 32):
    q, k = torch.randn(t, 8), torch.randn(t, 8)
    print(f'   T={t:4d}    ->  scores {tuple((q @ k.T).shape)}   ({(q @ k.T).numel()} entries)')

vary d_k, hold T=4:
   d_k=   8  ->  scores (4, 4)   (16 entries)
   d_k=  64  ->  scores (4, 4)   (16 entries)
   d_k= 512  ->  scores (4, 4)   (16 entries)

vary T, hold d_k=8:
   T=   4    ->  scores (4, 4)   (16 entries)
   T=   8    ->  scores (8, 8)   (64 entries)
   T=  16    ->  scores (16, 16)   (256 entries)
   T=  32    ->  scores (32, 32)   (1024 entries)


### Reading the output
`d_k` from 8 to 512 — the score matrix stays `(4, 4)`, 16 entries.
`T` doubling from 4 to 32 — entries go 16 → 64 → 256 → 1024. **Quadratic.**

This is the whole reason long context is hard, and it is a *memory* argument before it is a
compute one.

## Experiment 6 — put a number on it (interview hook)

A real serving configuration: batch 8, 32 heads, context 8192, fp16 (2 bytes).
Every head keeps its own `(T, T)` score matrix.

In [7]:
B, H, T_ctx, bytes_per = 8, 32, 8192, 2

entries = B * H * T_ctx * T_ctx
total   = entries * bytes_per

print(f'entries  = B*H*T*T = {B}*{H}*{T_ctx}*{T_ctx} = {entries:,}')
print(f'bytes    = {total:,}')
print(f'         = {total / 1024**3:.1f} GiB   <- for ONE attention layer, one forward pass')
print()
for t in (2048, 4096, 8192, 16384):
    g = B * H * t * t * bytes_per / 1024**3
    print(f'   T={t:6d}  ->  {g:8.1f} GiB')

entries  = B*H*T*T = 8*32*8192*8192 = 17,179,869,184
bytes    = 34,359,738,368
         = 32.0 GiB   <- for ONE attention layer, one forward pass

   T=  2048  ->       2.0 GiB
   T=  4096  ->       8.0 GiB
   T=  8192  ->      32.0 GiB
   T= 16384  ->     128.0 GiB


### Reading the output
**32 GiB** — for one layer, in one forward pass, on an 80 GB GPU. Multiply by the layer count
and it is absurd. Double the context and it quadruples.

Nobody materialises that matrix. FlashAttention's entire contribution is computing attention
**without ever writing the `(T, T)` matrix to memory** — it tiles the computation and keeps a
running softmax instead. You will build the online-softmax recurrence by hand in Phase 6.

When an interviewer asks *"why is long context hard"*, this number is the answer.

## Experiment 7 — the bill for a free transpose

Transpose costs nothing because it only swaps strides. The cost arrives later, when something
needs contiguous memory.

In [8]:
A = torch.arange(6).reshape(2, 3)

print('A.view(3, 2)   ->', tuple(A.view(3, 2).shape), ' fine, A is contiguous')

try:
    A.T.view(2, 3)
except RuntimeError as e:
    print('A.T.view(2, 3) -> RuntimeError:', str(e).split('\n')[0][:90])

print('A.T.reshape(2, 3) ->', tuple(A.T.reshape(2, 3).shape), ' works — reshape copies when it must')
print('A.T.contiguous().view(2, 3) ->', tuple(A.T.contiguous().view(2, 3).shape), ' works — you paid for the copy')

A.view(3, 2)   -> (3, 2)  fine, A is contiguous
A.T.view(2, 3) -> RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension sp
A.T.reshape(2, 3) -> (2, 3)  works — reshape copies when it must
A.T.contiguous().view(2, 3) -> (2, 3)  works — you paid for the copy


### Reading the output
`.view()` refuses on the transposed tensor; `.reshape()` quietly copies instead; `.contiguous()`
makes the copy explicit.

This exact trap is what scrambles multi-head attention heads in Phase 3 — a reshape that
"works" but silently reorders your data. 01.5 is the lesson that takes it apart.

---
## Challenge — predict first, then run

Write your answers before running.

| # | Question | Your answer |
| --- | --- | --- |
| 1 | `X` is `(6, 5)`. Shape and stride of `X.T`? | _(write)_ |
| 2 | `Q` is `(10, 64)`, `K` is `(10, 64)`. Shape of `Q @ K.T`? | _(write)_ |
| 3 | Same `Q`, `K` but `d_k = 4096`. Shape of `Q @ K.T`? | _(write)_ |
| 4 | `A` is `(3, 4)`, `B` is `(4, 2)`. Shape of `(A @ B).T`? And of `B.T @ A.T`? | _(write)_ |
| 5 | Does `A.T.is_contiguous()` return True or False for a `(3, 4)` tensor? | _(write)_ |

And in words: score-matrix memory at `B=4, H=16, T=4096, fp16` — is it bigger or smaller than
the 32 GiB above, and by what factor? Work it out before running.

In [ ]:
# run AFTER writing your predictions above
X = torch.randn(6, 5)
print('1.', tuple(X.T.shape), X.T.stride())

for d in (64, 4096):
    q, k = torch.randn(10, d), torch.randn(10, d)
    print(f'2/3. d_k={d:5d} ->', tuple((q @ k.T).shape))

A, Bm = torch.randn(3, 4), torch.randn(4, 2)
print('4.', tuple((A @ Bm).T.shape), 'and', tuple((Bm.T @ A.T).shape))

print('5.', torch.randn(3, 4).T.is_contiguous())

g = 4 * 16 * 4096 * 4096 * 2 / 1024**3
print(f'\nB=4,H=16,T=4096,fp16 -> {g:.1f} GiB')